# Laos IREF Project Hub

This notebook is the project control panel for the Laos IREF paper. It sits on top of the existing Python and R pipeline, lets you validate the core paths, rerun the full workflow from one place, and preview the outputs that feed the replication package.

## What to do here
1. Confirm the datasets, results summary, and output paths.
2. Set `RUN_PIPELINE = True` if you want to refresh the full analysis.
3. Inspect the current output manifest, key tables, and figures.
4. Use the quick links table at the end to jump into the results summary, datasets, and output folders.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate the project root containing pyproject.toml.")
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

pd.options.display.max_columns = 50
pd.options.display.float_format = lambda value: f"{value:,.3f}"
plt.style.use("seaborn-v0_8-whitegrid")

from laos_fx_oil_macro.notebook_helpers import (
    build_notebook_project,
    load_csv,
    output_manifest_table,
    path_status_table,
    quick_links_table,
    round_numeric_columns,
    run_pipeline_from_notebook,
)

project = build_notebook_project(PROJECT_ROOT)
project.root


In [ ]:
path_status = path_status_table(project)
display(path_status)


In [ ]:
RUN_PIPELINE = False
PIPELINE_PARAMS = {
    "tvp_draws": 1000,
    "tvp_burnin": 300,
    "tvp_thin": 2,
    "tvp_horizons": 12,
    "lp_horizons": 12,
    "lp_lags": 2,
    "skip_tvp": False,
}
PIPELINE_PARAMS


In [ ]:
manifest = None
if RUN_PIPELINE:
    manifest = run_pipeline_from_notebook(project, **PIPELINE_PARAMS)
    manifest_table = pd.DataFrame(
        {
            "artifact": manifest.keys(),
            "path": [str(path.relative_to(project.root)) for path in manifest.values()],
        }
    )
    display(manifest_table)
else:
    print("RUN_PIPELINE is False. Loading the current output directory instead.")


In [ ]:
manifest_table = output_manifest_table(project)
display(manifest_table)


In [ ]:
preview_paths = {
    "processed_panel": project.processed_panel,
    "break_diagnostics": project.tables_dir / "break_diagnostics.csv",
    "robustness_summary": project.tables_dir / "robustness_summary.csv",
    "tvp_fit_summary": project.models_dir / "tvp_svar_fit_summary.csv",
}

for label, path in preview_paths.items():
    if not path.exists():
        print(f"Missing: {path.relative_to(project.root)}")
        continue
    print(label)
    preview = load_csv(path).head(8)
    display(round_numeric_columns(preview))


In [ ]:
for figure_name in ["macro_context.png", "lp_fx_to_inflation.png", "crisis_overlay.png"]:
    figure_path = project.figures_dir / figure_name
    if figure_path.exists():
        print(figure_path.relative_to(project.root))
        display(Image(filename=str(figure_path), width=820))


In [ ]:
quick_links = quick_links_table(project)
display(quick_links)
